<a href="https://colab.research.google.com/github/darshanlahamage/Adaptive-Adversarial-Augmentation/blob/main/vg_adversarial_aug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 || true
!pip install --quiet scikit-learn matplotlib tqdm

In [ ]:
!pip install --upgrade torch torchvision torchaudio

In [ ]:
import os, math, time, random, json
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models, utils as vutils
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

def get_device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

device = get_device()
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

In [ ]:
def compute_ece(probs, correct, n_bins=15):
    bin_edges = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    n = probs.shape[0]
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        acc = correct[mask].mean()
        conf = probs[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)

def compute_failure_diversity(features, pred_labels, true_labels, n_clusters=10, pca_dim=64, random_state=0):
    mask = pred_labels != true_labels
    if mask.sum() == 0:
        return 0.0, 0
    X = features[mask]
    n_samples = X.shape[0]
    pca_dim = min(pca_dim, X.shape[1])
    if n_samples <= 2:
        return 0.0, int(n_samples)
    pca = PCA(n_components=pca_dim, random_state=random_state)
    Xr = pca.fit_transform(X)
    k = min(n_clusters, max(2, n_samples // 10))
    kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10).fit(Xr)
    counts = np.bincount(kmeans.labels_, minlength=k)
    probs = counts / counts.sum()
    ent = -np.sum([p * math.log(p + 1e-12) for p in probs if p > 0])
    max_ent = math.log(k)
    return float(ent / (max_ent + 1e-12)), int(n_samples)

In [ ]:
class ResNetFeatureWrapper(nn.Module):
    def __init__(self, num_classes=10, pretrained=False):
        super().__init__()
        # Updated to use modern torchvision weights API
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        self.model = models.resnet18(weights=weights)
        in_feats = self.model.fc.in_features
        self.model.fc = nn.Linear(in_feats, num_classes)

    def forward(self, x):
        return self.model(x)

    def extract_features(self, x):
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)
        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)
        x = self.model.avgpool(x)
        return torch.flatten(x, 1)

In [ ]:
class TransformGenerator(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(3, hidden, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        # Reduced to 5 strictly differentiable parameters
        self.fc = nn.Linear(hidden, 5)

    def forward(self, x):
        z = torch.flatten(self.encoder(x), 1)
        params = self.fc(z)

        rot = torch.tanh(params[:, 0]) * 20  # +/- 20 degrees
        tx  = torch.tanh(params[:, 1]) * 4   # +/- 4 pixels
        ty  = torch.tanh(params[:, 2]) * 4
        brightness = torch.tanh(params[:, 3]) * 0.2
        contrast   = 1 + torch.tanh(params[:, 4]) * 0.2

        return rot, tx, ty, brightness, contrast

def spatial_transform(x, rot, tx, ty):
    B, C, H, W = x.shape
    theta = torch.zeros(B, 2, 3).to(x.device)
    angle = rot * math.pi / 180

    theta[:, 0, 0] = torch.cos(angle)
    theta[:, 0, 1] = -torch.sin(angle)
    theta[:, 1, 0] = torch.sin(angle)
    theta[:, 1, 1] = torch.cos(angle)
    theta[:, 0, 2] = tx / W
    theta[:, 1, 2] = ty / H

    grid = F.affine_grid(theta, x.size(), align_corners=False)
    # Using reflection padding to avoid harsh black edges that the classifier could exploit
    return F.grid_sample(x, grid, align_corners=False, padding_mode='reflection')

def photometric_transform(x, brightness, contrast):
    x = x * contrast.view(-1, 1, 1, 1)
    x = x + brightness.view(-1, 1, 1, 1)
    return torch.clamp(x, 0, 1)

def apply_generator(x, params):
    rot, tx, ty, b, c = params
    x = spatial_transform(x, rot, tx, ty)
    x = photometric_transform(x, b, c)
    return x

In [ ]:
cifar_mean = np.array([0.4914, 0.4822, 0.4465])
cifar_std  = np.array([0.2470, 0.2435, 0.2616])

def get_cifar10_loaders(data_dir="./data", batch_size=128, num_workers=4):
    train_transforms = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(cifar_mean, cifar_std)
    ])
    test_transforms = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cifar_mean, cifar_std)
    ])

    train_ds = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=train_transforms)
    test_ds  = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=test_transforms)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

def save_image_grid(tensor_list, filename, nrow=8, pad_value=0.03):
    if not tensor_list: return
    norm_stack = torch.stack(tensor_list, dim=0)
    mean = torch.tensor(cifar_mean).view(1,3,1,1).to(norm_stack.device)
    std  = torch.tensor(cifar_std).view(1,3,1,1).to(norm_stack.device)
    unnorm = torch.clamp(norm_stack * std + mean, 0.0, 1.0)
    grid = vutils.make_grid(unnorm, nrow=nrow, padding=2, pad_value=pad_value)
    vutils.save_image(grid, filename)

In [ ]:
def minimax_train_epoch(train_loader, model, gen, optim_clf, optim_gen, criterion,
                        epoch, device, save_dir, k_classifier_steps=1,
                        lambda_reg=5.0, topk_save=8):
    ensure_dir(save_dir)
    stats = {"loss_clf": [], "loss_aug": [], "reg_feat": []}
    gallery_items = []

    mean = torch.tensor(cifar_mean, dtype=torch.float).view(1,3,1,1).to(device)
    std  = torch.tensor(cifar_std, dtype=torch.float).view(1,3,1,1).to(device)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False)

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        images_01 = images * std + mean

        # --- OPTIMIZATION 1: Extract clean anchor features ONCE at the start ---
        model.eval()
        with torch.no_grad():
            _, feats_orig = model.forward_with_features(images)
            feats_orig = feats_orig.detach()

        # --- STEP 1: Classifier Update ---
        model.train()
        gen.eval()
        loss_clf_val = 0

        for _ in range(k_classifier_steps):
            optim_clf.zero_grad()
            with torch.no_grad():
                params = gen(images_01)
                images_aug_01 = apply_generator(images_01, params)
                images_aug = (images_aug_01 - mean) / std

            logits = model(images_aug)
            loss = criterion(logits, labels)
            loss.backward()
            optim_clf.step()
            loss_clf_val = loss.item()

        stats["loss_clf"].append(loss_clf_val)

        # --- STEP 2: Generator Update ---
        gen.train()
        optim_gen.zero_grad()

        # Freeze classifier to prevent gradient leakage
        for param in model.parameters():
            param.requires_grad = False

        # Generate adversarial augmentations
        params = gen(images_01)
        images_aug_01 = apply_generator(images_01, params)
        images_aug = (images_aug_01 - mean) / std

        # --- OPTIMIZATION 2: Single forward pass for BOTH logits and features ---
        model.eval() # Keep in eval to match anchor BN stats
        logits_aug, feats_aug = model.forward_with_features(images_aug)

        loss_aug = criterion(logits_aug, labels)
        reg_feat = torch.mean((feats_aug - feats_orig) ** 2)

        # Generator maximizes classification loss, minimizes feature distance
        gen_loss = -loss_aug + lambda_reg * reg_feat
        gen_loss.backward()

        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)
        optim_gen.step()

        # Unfreeze classifier for the next batch
        for param in model.parameters():
            param.requires_grad = True

        stats["loss_aug"].append(loss_aug.item())
        stats["reg_feat"].append(reg_feat.item())

        if epoch % 5 == 0 or epoch == 1:
            for i in range(min(topk_save, images_aug.size(0))):
                gallery_items.append((loss_aug.item(), images_aug[i].detach().cpu()))

        pbar.set_postfix({
            "clf_loss": f"{loss_clf_val:.4f}",
            "gen_loss": f"{loss_aug.item():.4f}",
            "feat_reg": f"{reg_feat.item():.5f}"
        })

    if gallery_items:
        gallery_items_sorted = sorted(gallery_items, key=lambda x: -x[0])[:64]
        tensors = [t for (_, t) in gallery_items_sorted]
        save_image_grid(tensors, os.path.join(save_dir, f"gallery_epoch{epoch}.png"), nrow=8)

    return {k: float(np.mean(v)) for k, v in stats.items()}

In [ ]:
def evaluate_model(model, loader, device, return_features=False):
    model.to(device)
    model.eval()
    total, correct = 0, 0
    confs, correct_flags = [], []
    all_feats, all_preds, all_trues = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            probs = F.softmax(logits, dim=1)
            conf, pred = probs.max(dim=1)
            correct_batch = pred.eq(labels)

            total += images.size(0)
            correct += correct_batch.sum().item()
            confs.append(conf.cpu().numpy())
            correct_flags.append(correct_batch.cpu().numpy())

            if return_features:
                all_feats.append(model.extract_features(images).cpu().numpy())
                all_preds.append(pred.cpu().numpy())
                all_trues.append(labels.cpu().numpy())

    acc = correct / total
    confs, correct_flags = np.concatenate(confs), np.concatenate(correct_flags)
    ece = compute_ece(confs, correct_flags)

    if return_features:
        return acc, ece, np.concatenate(all_feats, axis=0), np.concatenate(all_preds, axis=0), np.concatenate(all_trues, axis=0)
    return acc, ece

def run_smoke_test(epochs=30, batch_size=128, k_classifier_steps=1,
                   gen_lr=1e-4, clf_lr=0.05, lambda_reg=5.0,
                   save_dir="./runs_v2/adaptive_aug", device=device):
    ensure_dir(save_dir)
    train_loader, test_loader = get_cifar10_loaders("./data", batch_size=batch_size)

    model = ResNetFeatureWrapper(num_classes=10).to(device)
    gen = TransformGenerator().to(device)

    # Note: Weight decay helps stabilize adversarial training
    optimizer_clf = optim.SGD(model.parameters(), lr=clf_lr, momentum=0.9, weight_decay=5e-4)
    optimizer_gen = optim.Adam(gen.parameters(), lr=gen_lr)

    # Scheduler for the classifier
    scheduler_clf = optim.lr_scheduler.CosineAnnealingLR(optimizer_clf, T_max=epochs)
    criterion = nn.CrossEntropyLoss()
    history = defaultdict(list)

    for epoch in range(1, epochs+1):
        t0 = time.time()
        agg = minimax_train_epoch(train_loader, model, gen, optimizer_clf, optimizer_gen, criterion,
                                  epoch, device, save_dir,
                                  k_classifier_steps=k_classifier_steps,
                                  lambda_reg=lambda_reg, topk_save=8)
        scheduler_clf.step()

        acc, ece = evaluate_model(model, test_loader, device, return_features=False)
        history['train_loss'].append(agg['loss_clf'])
        history['train_loss_aug'].append(agg['loss_aug'])
        history['reg_feat'].append(agg['reg_feat'])
        history['test_acc'].append(acc)
        history['test_ece'].append(ece)

        print(f"Epoch {epoch} | Time: {time.time()-t0:.1f}s | Clf Loss: {agg['loss_clf']:.4f} | Gen Loss: {agg['loss_aug']:.4f} | Feat Reg: {agg['reg_feat']:.5f} | Test Acc: {acc:.4f} | ECE: {ece:.4f}")

    # Final detailed eval
    acc, ece, feats, pred_arr, true_arr = evaluate_model(model, test_loader, device, return_features=True)
    fds, n_fail = compute_failure_diversity(feats, pred_arr, true_arr, n_clusters=10)
    print(f"\nFinal Metrics -> Acc: {acc:.4f} | ECE: {ece:.4f} | Failures: {n_fail} | FDS: {fds:.4f}")

    torch.save({'model_state': model.state_dict(), 'gen_state': gen.state_dict()}, os.path.join(save_dir, 'final_ckpt.pth'))
    with open(os.path.join(save_dir, 'history.json'), 'w') as f:
        json.dump({k: list(v) for k,v in history.items()}, f)

    return model, gen, history

# Execute the test
model, gen, history = run_smoke_test(epochs=30, batch_size=128, k_classifier_steps=1, gen_lr=1e-4, clf_lr=0.05, lambda_reg=5.0, save_dir="./runs_v2/adaptive_aug", device=device)